In [0]:
%python
import random
from datetime import datetime, timedelta
from pyspark.sql import Row

RAW_PATH = "/Volumes/workspace/rp_landing/raw"

# Stores - 5
stores = []
cities = ["Mumbai", "Pune", "Delhi", "Bangalore", "Indore"]
for i in range(1, 6):
    stores.append(Row(store_id=f"S{i:03d}", store_name=f"Store {i}", city=cities[i-1], state="MH"))
spark.createDataFrame(stores).coalesce(1).write.mode("overwrite").option("header", True).csv(f"{RAW_PATH}/stores")

# Products - 20
products = []
for i in range(1, 21):
    products.append(Row(product_id=f"P{i:03d}", product_name=f"Product {i}", category=random.choice(["Grocery","Dairy","Bakery","Beverages"]), unit_price=round(random.uniform(20, 500), 2)))
spark.createDataFrame(products).coalesce(1).write.mode("overwrite").option("header", True).csv(f"{RAW_PATH}/products")

# Sales - 500 rows
sales = []
for i in range(500):
    sales.append(Row(sale_date=(datetime.now() - timedelta(days=random.randint(0, 30))).strftime("%Y-%m-%d"), store_id=f"S{random.randint(1,5):03d}", product_id=f"P{random.randint(1,20):03d}", qty=random.randint(1, 10), amount=round(random.uniform(50, 2000), 2)))
spark.createDataFrame(sales).coalesce(1).write.mode("overwrite").option("header", True).csv(f"{RAW_PATH}/sales_daily")

print("M1 DONE")

In [0]:
SELECT count(*) FROM workspace.rp_bronze.sales_daily

In [0]:
%python
release(OUT, days=63)

In [0]:
SELECT 
  max(sale_date) as last_date,
  count(*) as total,
  count(promo_flag) as promo_count,
  count(channel) as channel_count
FROM workspace.rp_bronze.sales

In [0]:
SELECT 'sales' as table_name, count(*) as total FROM workspace.rp_bronze.sales
UNION ALL
SELECT 'stores', count(*) FROM workspace.rp_bronze.stores
UNION ALL
SELECT 'products', count(*) FROM workspace.rp_bronze.products
UNION ALL
SELECT 'sales_history', count(*) FROM workspace.rp_bronze.sales_history

In [0]:
%python
release(OUT, days=63)

In [0]:
SELECT min(sale_date) as start_date, max(sale_date) as last_date, count(*) as total
FROM workspace.rp_bronze.sales_daily

In [0]:
SHOW TABLES IN workspace.rp_bronze

In [0]:
SELECT * FROM workspace.rp_bronze.sales_daily LIMIT 2

In [0]:
DESCRIBE TABLE workspace.rp_bronze.sales_daily

In [0]:
SELECT 
  count(*) as total_rows,
  sum(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) as rescued_rows,
  sum(CASE WHEN promo_flag IS NOT NULL THEN 1 ELSE 0 END) as promo_flag_rows,
  sum(CASE WHEN channel IS NOT NULL THEN 1 ELSE 0 END) as channel_rows
FROM workspace.rp_bronze.sales_daily

In [0]:
DESCRIBE HISTORY workspace.rp_bronze.sales_daily

In [0]:
SELECT 'sales_daily' as tbl, sum(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) as rescued FROM workspace.rp_bronze.sales_daily
UNION ALL
SELECT 'planner_overrides', sum(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) FROM workspace.rp_bronze.planner_overrides
UNION ALL
SELECT 'forecast_runs', sum(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) FROM workspace.rp_bronze.forecast_runs

In [0]:
SELECT version, timestamp, operation, operationMetrics.numOutputRows as rows_written
FROM (DESCRIBE HISTORY workspace.rp_bronze.sales_daily)
ORDER BY version

In [0]:
%python
# Import generator file as Python module (not with %run)
import sys
sys.path.append("/Workspace/Users/vaibhavirai444@gmail.com")

from retail_data_generator import Generator, release

print("Generator loaded successfully")
print(f"release function: {release}")

In [0]:
%python
# Release remaining weeks to reach full history (142 files)
OUT = "/Volumes/workspace/rp_landing/raw"
BASE = "/Volumes/workspace/rp_landing/raw/sales_daily"

import os

print(f"Starting files: {len(os.listdir(BASE))}")

while len(os.listdir(BASE)) < 142:
    release(OUT, days=7)
    print(f"Files now: {len(os.listdir(BASE))}")

print("Done - Full history released")
print(f"Final files: {len(os.listdir(BASE))}")

In [0]:
%python
# Stop loop - check why release thinks it is already done
import inspect
from retail_data_generator import release

print(inspect.getsource(release))

In [0]:
%python
# Check what is actually inside Raw folder
import os

OUT = "/Volumes/workspace/rp_landing/raw"

for root, dirs, files in os.walk(OUT):
    print(f"Folder: {root}")
    print(f"  Subfolders: {dirs}")
    print(f"  Files count: {len(files)}")
    for f in sorted(files)[:5]:
        print(f"    - {f}")
    print("")

In [0]:
%python
# Check correct Raw landing folder (where release actually copies to)
import os

OUT = "/Volumes/workspace/rp_landing/raw"

for folder in [
    f"{OUT}/landing/sales_daily",
    f"{OUT}/_source/sales_daily",
    f"{OUT}/landing",
]:
    print(f"\nFolder: {folder}")
    if os.path.exists(folder):
        files = sorted(os.listdir(folder))
        print(f"  Count: {len(files)}")
        for f in files[:10]:
            print(f"  - {f}")
    else:
        print("  DOES NOT EXIST")

In [0]:
%python
# Reset state and release to correct landing folder
import json
import os

OUT = "/Volumes/workspace/rp_landing/raw"
BASE = "/Volumes/workspace/rp_landing/raw/landing/sales_daily"
STATE_PATH = f"{OUT}/_release_state.json"

# Check how many you already have in landing
existing = len(os.listdir(BASE)) if os.path.exists(BASE) else 0
print(f"Existing in landing: {existing}")

# Reset state to match existing (e.g. if 12 files, reset to 11)
# If existing is 0, reset to -1
reset_to = existing - 1
with open(STATE_PATH, "w") as f:
    json.dump({"released_through_day": reset_to}, f)

print(f"State reset to {reset_to}")

for i in range(25):
    release(OUT, days=7)
    count = len(os.listdir(BASE))
    print(f"Batch {i+1}: Files now: {count}")
    if count >= 142:
        break

print(f"Final files in landing: {len(os.listdir(BASE))}")

In [0]:
SELECT 'stores' AS table_name, COUNT(*) AS rows FROM workspace.rp_bronze.stores
UNION ALL
SELECT 'products', COUNT(*) FROM workspace.rp_bronze.products
UNION ALL
SELECT 'sales_history', COUNT(*) FROM workspace.rp_bronze.sales_history
UNION ALL
SELECT 'forecast_runs', COUNT(*) FROM workspace.rp_bronze.forecast_runs
UNION ALL
SELECT 'sales_daily', COUNT(*) FROM workspace.rp_bronze.sales_daily
UNION ALL
SELECT 'planner_overrides', COUNT(*) FROM workspace.rp_bronze.planner_overrides;

In [0]:
SELECT _source_file, COUNT(*) AS rows
FROM workspace.rp_bronze.products
GROUP BY _source_file
ORDER BY _source_file;

In [0]:
DELETE FROM workspace.rp_bronze.products
WHERE _source_file NOT LIKE '%/landing/%';

In [0]:
SELECT COUNT(*) FROM workspace.rp_bronze.products;

In [0]:
SELECT _source_file, COUNT(*) AS rows
FROM workspace.rp_bronze.stores
GROUP BY _source_file
ORDER BY _source_file;

In [0]:
DELETE FROM workspace.rp_bronze.stores
WHERE _source_file NOT LIKE '%/landing/%';

In [0]:
SELECT COUNT(*) FROM workspace.rp_bronze.stores;

In [0]:
SELECT 'stores' AS table_name, COUNT(*) AS rows FROM workspace.rp_bronze.stores
UNION ALL
SELECT 'products', COUNT(*) FROM workspace.rp_bronze.products
UNION ALL
SELECT 'sales_history', COUNT(*) FROM workspace.rp_bronze.sales_history
UNION ALL
SELECT 'forecast_runs', COUNT(*) FROM workspace.rp_bronze.forecast_runs
UNION ALL
SELECT 'sales_daily', COUNT(*) FROM workspace.rp_bronze.sales_daily
UNION ALL
SELECT 'planner_overrides', COUNT(*) FROM workspace.rp_bronze.planner_overrides;